In [1]:
import yaml
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis.parsimonious import optimize_minimal_flux

In [55]:
cell_type = "MCF7_core"

sbml_file = "../config/MCF7_Zielinski_2017_core.sbml"
model = read_sbml_model(sbml_file, f_replace={'F_REACTION': lambda x: x})
model.solver = "cplex"
exchanges = [r.id for r in model.exchanges if r.id.startswith("R_EX")]

for r in model.reactions:
    if r.objective_coefficient != 0:
        objective_reaction = r.id
        break

model.solver

In [23]:
for r in model.reactions:
    if r.bounds[0] > 0:
        print(r.id, r.bounds)
        continue
    if r.bounds[1] < 0:
        print(r.id, r.bounds)
        continue


for r in model.exchanges:
    if r.lower_bound < 0:
        print(r.id, r.bounds)
        r.lower_bound = -0.02
        continue
    if r.upper_bound < 0:
        print(r.id, r.bounds)
        continue
    if r.lower_bound > 0:
        print(r.id, r.bounds)
        continue   

R_ATPM (1.07, 1000.0)
R_EX_arg_L_e (-0.02, 0.0)
R_EX_asn_L_e (-0.02, 0.0)
R_EX_asp_L_e (-0.02, 0.0)
R_EX_chol_e (-0.02, 1000.0)
R_EX_cl_e (-0.02, 1000.0)
R_EX_glc_e (-0.02, 0.0)
R_EX_gln_L_e (-0.02, 0.0)
R_EX_glu_L_e (-0.02, 0.0)
R_EX_gly_e (-0.02, 0.0)
R_EX_h_e (-0.02, 1000.0)
R_EX_h2o_e (-0.02, 1000.0)
R_EX_ile_L_e (-0.02, 0.0)
R_EX_k_e (-0.02, 1000.0)
R_EX_leu_L_e (-0.02, 0.0)
R_EX_lys_L_e (-0.02, 0.0)
R_EX_na1_e (-0.02, 1000.0)
R_EX_nh4_e (-0.02, 1000.0)
R_EX_o2_e (-0.02, 0.0)
R_EX_orn_e (-0.02, 0.0)
R_EX_phe_L_e (-0.02, 0.0)
R_EX_pi_e (-0.02, 1000.0)
R_EX_ser_L_e (-0.02, 0.0)
R_EX_thr_L_e (-0.02, 0.0)
R_EX_trp_L_e (-0.02, 0.0)
R_EX_tyr_L_e (-0.02, 0.0)
R_EX_val_L_e (-0.02, 0.0)


In [65]:
print(f"Objective reaction: {objective_reaction}")

model.reactions.R_DM_gudac_c_.upper_bound = 0.0
model.reactions.R_biomass_reaction.lower_bound = 0.0
model.reactions.R_biomass_reaction.upper_bound = 10
model.reactions.R_DM_atp_c_.lower_bound = 0.0
model.reactions.R_DM_atp_c_.upper_bound = 100
model.reactions.R_ATPM.lower_bound = 1.07

model.reactions.R_EX_lac_L_e.upper_bound = 10.0
model.reactions.R_EX_lac_L_e.lower_bound = 0.0

model.reactions.R_EX_glc_e.upper_bound = 0.0
model.reactions.R_EX_glc_e.lower_bound = -10.0

model.reactions.R_EX_pi_e.lower_bound = -100

model.reactions.R_EX_o2_e.lower_bound = -100

model.reactions.R_EX_ala_L_e.upper_bound = 0.0
model.reactions.R_EX_cit_e.upper_bound = 0.0
model.reactions.R_EX_pro_L_e.upper_bound = 0.0
model.reactions.R_EX_pro_L_e.lower_bound = 0.0

solution = model.optimize()
max_growth_rate = solution.objective_value
print(f"Max growth rate: {max_growth_rate} 1/hr")
print()
summary = model.summary(solution)
summary

Objective reaction: R_biomass_reaction
Max growth rate: 0.02205002837266478 1/hr



Metabolite,Reaction,Flux,C-Number,C-Flux
M_Tyr_ggn_c,R_DM_Tyr_ggn_c_,0.000756,0,0.00%
M_gudac_c,R_DM_gudac_c_,0.002282,3,0.53%
M_arg_L_e,R_EX_arg_L_e,0.008448,6,3.91%
M_asn_L_e,R_EX_asn_L_e,0.004861,4,1.50%
M_chol_e,R_EX_chol_e,0.0007798,5,0.30%
M_glc_D_e,R_EX_glc_e,0.09958,6,46.13%
M_gln_L_e,R_EX_gln_L_e,0.04151,5,16.02%
M_gly_e,R_EX_gly_e,0.006152,2,0.95%
M_ile_L_e,R_EX_ile_L_e,0.007143,6,3.31%
M_leu_L_e,R_EX_leu_L_e,0.01226,6,5.68%


In [27]:
def flux(X, Km = 1.0, Vmax = 0.6):
    return (X*Vmax)/(X+Km)

o2_exchange = model.reactions.R_EX_o2_e

o2_conc_threshold = 0.196525
lb = -1 * flux(o2_conc_threshold)

print(o2_conc_threshold, lb)
o2_exchange.lower_bound = 0

s = model.optimize()
model.summary(s)

0.196525 -0.09854787823070975


Metabolite,Reaction,Flux,C-Number,C-Flux
M_glc_D_e,R_EX_glc_e,0.535,6,100.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
M_h_e,R_EX_h_e,-1.07,0,0.00%
M_lac_L_e,R_EX_lac_L_e,-1.07,3,100.00%


In [74]:
# cell_type = "MCF7_core"

sbml_file = "../config/MCF7_Zielinski_2017_core_trimmed.sbml"
model = read_sbml_model(sbml_file, f_replace={'F_REACTION': lambda x: x})
model.solver = "cplex"



model.reactions.R_EX_arg_L_e.lower_bound = -0.008448
model.reactions.R_EX_asn_L_e.lower_bound = -0.004861
model.reactions.R_EX_asp_L_e.lower_bound = -0.000000
model.reactions.R_EX_chol_e.lower_bound = -0.000790

model.reactions.R_EX_co2_e.upper_bound = 0.944766
model.reactions.R_EX_gln_L_e.lower_bound = -0.041533
model.reactions.R_EX_glu_L_e.lower_bound = 0.000000
model.reactions.R_EX_gly_e.lower_bound = -0.006152
model.reactions.R_EX_ile_L_e.lower_bound = -0.007143

model.reactions.R_EX_lac_L_e.upper_bound = 10.000000
model.reactions.R_EX_lac_L_e.lower_bound = 0.0


model.reactions.R_EX_ala_L_e.lower_bound = 0.0 # 0.000000
model.reactions.R_EX_mal_L_e.upper_bound = 0.0 # 0.000399
model.reactions.R_EX_cit_e.upper_bound   = 0.0 # 0.002172

model.reactions.R_EX_urea_e.upper_bound  = 100.0 # 0.002634
model.reactions.R_EX_nh4_e.upper_bound   = 100.0 # 0.022598

model.reactions.R_EX_o2_e.lower_bound  = -0.9
model.reactions.R_EX_glc_e.lower_bound = -0.686008

model.reactions.R_EX_leu_L_e.lower_bound = -0.012255
model.reactions.R_EX_lys_L_e.lower_bound = -0.014071

model.reactions.R_EX_orn_e.lower_bound   = -0.000752
model.reactions.R_EX_phe_L_e.lower_bound = -0.003814
model.reactions.R_EX_pi_e.lower_bound    = -0.016359
model.reactions.R_EX_pro_L_e.lower_bound = -0.001349
model.reactions.R_EX_ser_L_e.lower_bound = -0.013819
model.reactions.R_EX_thr_L_e.lower_bound = -0.008443
model.reactions.R_EX_trp_L_e.lower_bound = -0.000890
model.reactions.R_EX_tyr_L_e.lower_bound = -0.003353

model.reactions.R_EX_val_L_e.lower_bound = -0.008611


sol = model.optimize()
s = None
if sol.status == 'optimal':
    print(sol.status)
    s = model.summary(s)
s  

optimal


Metabolite,Reaction,Flux,C-Number,C-Flux
M_Tyr_ggn_c,R_DM_Tyr_ggn_c_,0.0007657,0,0.00%
M_gudac_c,R_DM_gudac_c_,0.008832,3,2.03%
M_arg_L_e,R_EX_arg_L_e,0.008448,6,3.88%
M_asn_L_e,R_EX_asn_L_e,0.004861,4,1.49%
M_chol_e,R_EX_chol_e,0.0007898,5,0.30%
M_glc_D_e,R_EX_glc_e,0.0992,6,45.53%
M_gln_L_e,R_EX_gln_L_e,0.04153,5,15.88%
M_h_e,R_EX_h_e,0.01754,0,0.00%
M_ile_L_e,R_EX_ile_L_e,0.007143,6,3.28%
M_leu_L_e,R_EX_leu_L_e,0.01226,6,5.62%
